# SEIÐR — Scraping razas de gatos (TICA)
Fuente: https://tica.org/ticas-breeds/browse-all-breeds/
Objetivo: obtener 73 razas con nombre, temperamento, descripción y URL de imagen
para construir razas_tica_limpio.csv con escala Seiðr (0-5)

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd

options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
print('Selenium funcionando')

In [ ]:
# PASO 1: Obtener lista de razas con sus URLs
driver.get('https://tica.org/ticas-breeds/browse-all-breeds/')
time.sleep(3)

razas_lista = []
enlaces = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/breed/"]')

urls_vistas = set()
for a in enlaces:
    href = a.get_attribute('href')
    nombre = a.text.strip()
    if href and nombre and href not in urls_vistas:
        urls_vistas.add(href)
        razas_lista.append({'nombre': nombre, 'url': href})

print(f'Razas encontradas: {len(razas_lista)}')
for r in razas_lista[:5]:
    print(f'  {r["nombre"]} → {r["url"]}')

In [ ]:
# PASO 2: Explorar una ficha para entender su estructura
driver.get(razas_lista[0]['url'])
time.sleep(3)

# Ver todo el texto de la página
print(driver.find_element(By.TAG_NAME, 'body').text[:3000])

In [ ]:
# PASO 3: Explorar estructura HTML para encontrar los campos
# Buscar imagen principal
imgs = driver.find_elements(By.CSS_SELECTOR, 'img')
for img in imgs[:5]:
    print(f'src: {img.get_attribute("src")} | alt: {img.get_attribute("alt")}')

print()
# Buscar tags de temperamento (suelen estar en h2, h3 o párrafos destacados)
for tag in ['h1', 'h2', 'h3', 'h4']:
    elems = driver.find_elements(By.TAG_NAME, tag)
    for e in elems:
        print(f'{tag}: {e.text.strip()}')

In [ ]:
# PASO 4: Scraping completo de todas las razas
# AJUSTA los selectores CSS según lo que hayas visto en el paso anterior

datos_razas = []

for i, raza in enumerate(razas_lista):
    try:
        driver.get(raza['url'])
        time.sleep(2)

        # Nombre
        nombre = raza['nombre']

        # Imagen principal — ajusta el selector si es necesario
        url_imagen = ''
        try:
            img = driver.find_element(By.CSS_SELECTOR, '.wp-post-image, .breed-image img, article img')
            url_imagen = img.get_attribute('src')
        except:
            pass

        # Temperamento — suele estar en h2 o en el primer párrafo en mayúsculas
        temperamento = ''
        try:
            # Buscar el texto que suele ser INTELLIGENT • AFFECTIONATE etc.
            h2s = driver.find_elements(By.TAG_NAME, 'h2')
            for h2 in h2s:
                texto = h2.text.strip()
                if '•' in texto or texto.isupper():
                    temperamento = texto
                    break
        except:
            pass

        # Descripción — primer párrafo largo del artículo
        descripcion = ''
        try:
            parrafos = driver.find_elements(By.CSS_SELECTOR, 'article p, .entry-content p, .elementor-widget-text-editor p')
            for p in parrafos:
                texto = p.text.strip()
                if len(texto) > 100:
                    descripcion = texto
                    break
        except:
            pass

        datos_razas.append({
            'breed': nombre,
            'temperament': temperamento,
            'descripcion': descripcion,
            'url_imagen': url_imagen,
            'url_ficha': raza['url']
        })

        print(f'✓ {i+1}/{len(razas_lista)} {nombre}')

    except Exception as e:
        print(f'✗ {raza["nombre"]}: {e}')
        datos_razas.append({'breed': raza['nombre'], 'url_ficha': raza['url']})

driver.quit()
df_gatos = pd.DataFrame(datos_razas)
print(f'\nTotal: {len(df_gatos)} razas')
df_gatos.head()

In [ ]:
# PASO 5: Guardar CSV raw
df_gatos.to_csv('../universos/razas_tica_raw.csv', index=False, encoding='utf-8-sig')
print(f'Guardado: {len(df_gatos)} razas en razas_tica_raw.csv')
df_gatos[['breed', 'temperament', 'descripcion']].head(10)